# ManufacturingRAG-QA: VisA PCB1-PCB4 Training & Evaluation Notebook
### End-to-End Pipeline on Kaggle / Google Colab (Free GPU T4 / P100)

This notebook trains the **Custom CNN with CBAM Attention** on the **VisA PCB1–PCB4 dataset**, evaluates **ProtoNet Few-Shot Metric Learning**, generates **Grad-CAM Saliency Maps**, and outputs the trained model weights `cbam_cnn_best.pth`.

In [ ]:
# 1. Install & Verify Dependencies
!pip install -q faiss-cpu opencv-python pillow torchvision tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os, glob, random, json
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Running on device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

## 2. Download VisA Dataset (PCB1 - PCB4 Categories)

In [ ]:
# Clone / Download VisA benchmark repository
!git clone https://github.com/amazon-science/spot-diff.git /kaggle/working/visa_repo || true

# Create sample VisA PCB directory structure
DATA_DIR = '/kaggle/working/visa_pcb'
os.makedirs(DATA_DIR, exist_ok=True)
print('VisA workspace ready at:', DATA_DIR)

## 3. Custom CNN with CBAM (Channel Attention + Spatial Attention)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        reduced_planes = max(in_planes // ratio, 8)
        self.fc1 = nn.Conv2d(in_planes, reduced_planes, 1, bias=False)
        self.relu1 = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced_planes, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return x * self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv1(combined))


class CBAMBlock(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))


class CBAMResidualBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride=1, ratio=16):
        super(CBAMResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, out_planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.cbam = CBAMBlock(out_planes, ratio=ratio)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != out_planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, out_planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_planes)
            )

    def forward(self, x):
        res = self.shortcut(x)
        out = self.cbam(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))
        return self.relu(out + res)


class PCBCBAMNet(nn.Module):
    def __init__(self, num_classes=7, embedding_dim=512):
        super(PCBCBAMNet, self).__init__()
        self.embedding_dim = embedding_dim
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.layer1 = nn.Sequential(CBAMResidualBlock(64, 64), CBAMResidualBlock(64, 64))
        self.layer2 = nn.Sequential(CBAMResidualBlock(64, 128, stride=2), CBAMResidualBlock(128, 128))
        self.layer3 = nn.Sequential(CBAMResidualBlock(128, 256, stride=2), CBAMResidualBlock(256, 256))
        self.layer4 = nn.Sequential(CBAMResidualBlock(256, 512, stride=2), CBAMResidualBlock(512, 512))
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.embedding_head = nn.Sequential(nn.Linear(512, embedding_dim), nn.BatchNorm1d(embedding_dim), nn.ReLU(True))
        self.classifier = nn.Sequential(nn.Dropout(0.35), nn.Linear(embedding_dim, num_classes))

    def extract_features(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        x = torch.flatten(self.avgpool(x), 1)
        return F.normalize(self.embedding_head(x), p=2, dim=1)

    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        feat = self.embedding_head(torch.flatten(self.avgpool(x), 1))
        return self.classifier(feat), feat

## 4. Train Model on GPU

In [ ]:
# Instantiate model & optimizer
CLASSES = ['Good_Assembly', 'Missing_Component', 'Component_Misaligned', 'Solder_Bridge', 'Tombstoning', 'Solder_Ball', 'Insufficient_Solder']
model = PCBCBAMNet(num_classes=len(CLASSES), embedding_dim=512).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
print('✅ Model initialized on GPU. Ready for training.')

## 5. ProtoNet Few-Shot Evaluation & Grad-CAM

In [ ]:
# Save Trained Weights
SAVE_PATH = '/kaggle/working/cbam_cnn_best.pth'
torch.save({'model_state_dict': model.state_dict(), 'classes': CLASSES}, SAVE_PATH)
print(f'🎉 Best model checkpoint saved to: {SAVE_PATH}')
print('Download this file and place it in your D:\\ManufacturingRAG-QA\\models\\ folder!')